# PCA, NMF and anomaly detection — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D014_02_NMF_Decomposition.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# C4: Non-negative Matrix Factorization (NMF)

## 1. Introduction

**Non-negative Matrix Factorization (NMF)** is a dimension reduction technique similar to PCA, but with a strict constraint: **all values must be non-negative** (>= 0).

Because of this, NMF learns **parts-based representations**. Unlike PCA, which can have negative values (canceling things out), NMF is purely additive. This makes it incredibly useful for:
1.  **Recommender Systems**: Users' tastes are additive combinations of genres.
2.  **Topic Modeling**: Articles are additive combinations of topics.
3.  **Image Processing**: Faces/Digits are built from parts (eyes, noses / strokes).

### Concept
We decompose our data matrix **V** into two smaller matrices **W** (Features) and **H** (Weights/Components) such that $V \approx W \times H$.

![NMF Matrix Factorization](https://upload.wikimedia.org/wikipedia/commons/f/f9/NMF.png)
*(Source: Wikimedia Commons - NMF decomposing a matrix into two lower-rank non-negative matrices)*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize, MinMaxScaler

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# ═══ CHECKPOINT — "NMF for Recommender Systems (Music)" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize, MinMaxScaler
plt.rcParams['figure.figsize'] = (10, 6)

print("Checkpoint restored. Ready for: NMF for Recommender Systems (Music)")

## 2. NMF for Recommender Systems (Music)

We have a dataset of users and the artists they listen to. We want to recommend new artists.
- `scrobbler-small-sample.csv`: Contains `user_offset`, `artist_offset`, `playcount`.
- `artists.csv`: Contains artist names.

In [ ]:
# TODO: compute `df_scrobbler`, `df_artists`, `artist_names` from `Artist`
# Name your result: df_scrobbler, df_artists, artist_names, df_pivot

In [ ]:
# Apply NMF
# Let's look for, say, 10 latent features (Genres/Styles)
nmf_music = NMF(n_components=10, random_state=42)

# Fit to the matrix
W_music = nmf_music.fit_transform(X_music) # Artist Features
H_music = nmf_music.components_         # User Preferences

print("NMF Fitted!")

### Recommending
Now, if we pick an artist, we can find similar artists by looking at the `W` matrix (Artist Features). Artists with similar feature vectors are "musically similar".

In [ ]:
# TODO: compute `artist_index_map`, `target_artist`, `target_idx`
# Name your result: artist_index_map, target_artist, target_idx, target_features

## 3. Topic Modeling (Wikipedia)

NMF is fantastic for finding topics in text. We use `wikipedia-vectors.csv`, which is a Term-Document matrix (or Doc-Term, we'll check).

In [ ]:
# TODO: compute `df_wiki`, `articles`, `X_wiki`
# Name your result: df_wiki, articles, X_wiki, words

In [ ]:
# TODO: compute `nmf_wiki`, `H_wiki`, `top_indices`
# Name your result: nmf_wiki, H_wiki, top_indices, top_words

**Observation**: You should see clear themes! E.g., one topic might be ['film', 'movie', 'actor'], another ['climate', 'warming', 'gas'].

In [ ]:
# ═══ CHECKPOINT — "NMF for Images (Parts-Based Learning)" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize, MinMaxScaler
plt.rcParams['figure.figsize'] = (10, 6)
df_scrobbler = pd.read_csv('../../00-data/scrobbler-small-sample.csv')
df_artists = pd.read_csv('../../00-data/artists.csv', header=None, names=['Artist'])
artist_names = df_artists['Artist'].values
df_pivot = df_scrobbler.pivot(index='artist_offset', columns='user_offset', values='playcount')
df_pivot = df_pivot.fillna(0)
X_music = df_pivot.values
nmf_music = NMF(n_components=10, random_state=42)
W_music = nmf_music.fit_transform(X_music)
H_music = nmf_music.components_
artist_index_map = {name: i for i, name in enumerate(artist_names)}
target_artist = 'Massive Attack'
if target_artist in artist_index_map:
    target_idx = artist_index_map[target_artist]
    target_features = W_music[target_idx]
    similarities = W_music.dot(target_features)
    top_indices = similarities.argsort()[::-1][:6]
    print(f'Artists similar to {target_artist}:')
    for idx in top_indices:
        if idx != target_idx:
            print(f'- {artist_names[idx]}')
df_wiki = pd.read_csv('../../00-data/wikipedia-vectors.csv', index_col=0)
articles = df_wiki.columns.values
X_wiki = df_wiki.T.values
with open('../../00-data/wikipedia-vocabulary-utf8.txt', 'r') as f:
    words = f.read().splitlines()
nmf_wiki = NMF(n_components=6, random_state=42)
nmf_wiki.fit(X_wiki)
H_wiki = nmf_wiki.components_
for i, component in enumerate(H_wiki):
    top_indices = component.argsort()[::-1][:10]
    top_words = [words[k] for k in top_indices]
    print(f'Topic {i}: {top_words}')

print("Checkpoint restored. Ready for: NMF for Images (Parts-Based Learning)")

## 4. NMF for Images (Parts-Based Learning)

Finally, let's look at `lcd_digits.csv` again. NMF learns the "strokes" that make up the digits.

In [ ]:
# TODO: compute `df_digits`, `X_digits`, `nmf_digits`
# Name your result: df_digits, X_digits, nmf_digits, components

**Interpretation**: Unlike PCA eigen-digits (which look like ghostly complete digits), NMF components look like pieces: a vertical line, a loop, a horizontal bar.

## 5. Exercises

**Task**: Inspect the Wikipedia Topic Model results for a specific article.
1.  Transform the `X_wiki` using the fitted `nmf_wiki` to get the `W_wiki` (Article-Topic) matrix.
2.  Find the index for the article "Cristiano Ronaldo".
3.  Print his Topic weights. Which topic is he most associated with?

In [ ]:
# Solution Space
# Your code here
